<a href="https://colab.research.google.com/github/shemaiscard/Face-Analysis-pro/blob/main/faces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit deepface opencv-python-headless pillow pandas streamlit-option-menu streamlit-webrtc -q

In [ ]:
%%writefile app.py
import streamlit as st
import cv2
import numpy as np
from deepface import DeepFace
from PIL import Image
import os
import av
from streamlit_webrtc import webrtc_streamer, WebRtcMode, RTCConfiguration

# Suppress TF logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

st.set_page_config(page_title="Face AI", page_icon="", layout="wide")

# Custom UI Styling
st.markdown("""
    <style>
    .stApp { background-color: #0f172a; color: white; }
    h1, h2, h3 { color: #818cf8; }
    </style>
""", unsafe_allow_html=True)

st.title("Real-time Face Analysis Pro")
st.markdown("Analyze demographic attributes and emotions using state-of-the-art AI.")
st.divider()

# --- 1. SNAPSHOT FEATURE ---
st.subheader(" Option 1: High-Res Snapshot")
img_file = st.camera_input("Take a snapshot for detailed metrics")

if img_file:
    image = Image.open(img_file)
    np_img = np.array(image.convert("RGB"))

    with st.spinner('Analyzing...'):
        try:
            results = DeepFace.analyze(np_img, actions=["age", "gender", "emotion", "race"], enforce_detection=False)
            if not isinstance(results, list): results = [results]

            res = results[0] # Grab the first face detected

            c1, c2, c3, c4 = st.columns(4)
            gender = max(res['gender'].items(), key=lambda x: x[1])[0]
            c1.metric("Gender", gender)
            c2.metric("Age", f"~{int(res['age'])}")
            c3.metric("Emotion", res['dominant_emotion'].title())
            c4.metric("Race", res['dominant_race'].title())

        except Exception as e:
            st.error(f"Could not analyze image: {e}")

st.divider()

# --- 2. REAL-TIME VIDEO FEATURE ---
st.subheader(" Option 2: Live Video Stream")
st.markdown("Click **START** below to begin the live feed.")

# WebRTC needs STUN servers to navigate firewalls and connect your browser to Colab
RTC_CONFIGURATION = RTCConfiguration(
    {"iceServers": [{"urls": ["stun:stun.l.google.com:19302"]}]}
)

def process_video_frame(frame):
    # Convert WebRTC frame to OpenCV array
    img = frame.to_ndarray(format="bgr24")

    try:
        # Run DeepFace. enforce_detection=False prevents crashes when looking away
        results = DeepFace.analyze(img, actions=["age", "gender", "emotion"], enforce_detection=False)
        if not isinstance(results, list):
            results = [results]

        for res in results:
            x, y, w, h = res["region"]["x"], res["region"]["y"], res["region"]["w"], res["region"]["h"]

            # Draw bounding box
            cv2.rectangle(img, (x, y), (x+w, y+h), (255, 147, 211), 2)

            # Extract highest probability gender
            gender = max(res['gender'].items(), key=lambda x: x[1])[0]

            # Draw labels
            label = f"{gender} | {res['dominant_emotion']} | {int(res['age'])}"
            cv2.putText(img, label, (x, y-10), cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)

    except Exception:
        # If DeepFace fails (e.g., face moves out of frame), just return the untouched frame
        pass

    # Convert back to WebRTC format
    return av.VideoFrame.from_ndarray(img, format="bgr24")

# Streamlit WebRTC widget
webrtc_streamer(
    key="face-analysis",
    mode=WebRtcMode.SENDRECV,
    rtc_configuration=RTC_CONFIGURATION,
    video_frame_callback=process_video_frame,
    media_stream_constraints={"video": True, "audio": False},
    async_processing=True
)

In [ ]:
# 1. Install & Import
!pip install pyngrok -q
from pyngrok import ngrok
import os
import time
import subprocess

# 2. Setup Token
# Paste your token between the quotes below
CONF_TOKEN = "3AwVkv9LAYfcGjIe2UyoHU7gVt4_6E474cv7Y6t9ZzV8Y5Zsv"
os.system(f"ngrok config add-authtoken {CONF_TOKEN}")

# 3. Cleanup: Kill any old processes to prevent "port in use" errors
ngrok.kill()
os.system("pkill streamlit")

# 4. Start Streamlit in the background
# We pipe logs to a file so we can see if it fails
with open("streamlit_logs.txt", "w") as f:
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
                     stdout=f, stderr=f)

print("Starting Streamlit (waiting 8 seconds for TensorFlow to warm up)...")
time.sleep(8)

# 5. Connect Tunnel
try:
    # We specify 'addr' explicitly to avoid positional argument errors
    tunnel = ngrok.connect(8501)
    print("\n" + "🚀" * 15)
    print(f"SUCCESS! Click your link below:\n{tunnel.public_url}")
    print("🚀" * 15)
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    print("\nCheck Streamlit logs for clues:")
    !tail streamlit_logs.txt